# 🐔 PoultryVision AI — Chicken, Egg & Chick Detection

## Advanced Computer Vision Project

This notebook builds a practical **YOLO object-detection pipeline** for poultry monitoring. The scenario is a poultry farm where a camera captures chickens, eggs, and chicks and the AI system detects and counts them.

### Scenario
```text
Camera / Images
      ↓
YOLO Object Detection
      ↓
Chicken / Egg / Chick bounding boxes
      ↓
Confidence filtering
      ↓
Object counting
      ↓
Annotated output + metrics
```

### What you will learn
- YOLO dataset structure and validation
- Object detection training
- Transfer learning using pretrained YOLO weights
- Precision, Recall and mAP
- Confusion matrix / validation analysis
- Image inference
- Video/webcam inference
- Simple object counting
- Saving a production-ready model

> This notebook focuses on detection and counting. Multi-object tracking can be added later with ByteTrack or BoT-SORT.


In [ ]:
# Install dependencies
!pip -q install ultralytics opencv-python matplotlib pandas pyyaml

import os
import shutil
from pathlib import Path

import cv2
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ultralytics import YOLO

print('Ultralytics imported successfully.')

## 1. Dataset format

Use a YOLO-format dataset. A recommended structure is:

```text
poultry_dataset/
├── images/
│   ├── train/
│   ├── val/
│   └── test/
├── labels/
│   ├── train/
│   ├── val/
│   └── test/
└── poultry.yaml
```

Each image has a corresponding `.txt` label file. YOLO labels use:

`class_id x_center y_center width height`

with coordinates normalized from 0 to 1.


In [ ]:
# Change this to your dataset location.
# Example for Google Drive:
# DATASET_ROOT = '/content/drive/MyDrive/poultry_dataset'
DATASET_ROOT = '/content/poultry_dataset'

DATASET_ROOT = Path(DATASET_ROOT)
DATASET_ROOT.mkdir(parents=True, exist_ok=True)

print('Dataset root:', DATASET_ROOT)

In [ ]:
# Create a YOLO data configuration.
# Change class names to match your annotations.

CLASS_NAMES = ['chicken', 'egg', 'chick']

data_config = {
    'path': str(DATASET_ROOT),
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'names': {i: name for i, name in enumerate(CLASS_NAMES)},
}

YAML_PATH = DATASET_ROOT / 'poultry.yaml'
with open(YAML_PATH, 'w', encoding='utf-8') as f:
    yaml.safe_dump(data_config, f, sort_keys=False)

print(YAML_PATH.read_text())

## 2. Validate the dataset

Before training, check that image and label directories exist and that the number of images is reasonable. A production project should also validate class IDs, bounding-box ranges, missing labels, duplicates, and train/validation leakage.


In [ ]:
def count_files(folder, extensions):
    folder = Path(folder)
    if not folder.exists():
        return 0
    return sum(1 for p in folder.rglob('*') if p.suffix.lower() in extensions)

image_ext = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
label_ext = {'.txt'}

for split in ['train', 'val', 'test']:
    print(f'{split:>5}: images={count_files(DATASET_ROOT / "images" / split, image_ext)}, '
          f'labels={count_files(DATASET_ROOT / "labels" / split, label_ext)}')

In [ ]:
def validate_yolo_labels(split):
    label_dir = DATASET_ROOT / 'labels' / split
    problems = []
    if not label_dir.exists():
        return [f'Missing label directory: {label_dir}']

    for label_file in label_dir.glob('*.txt'):
        for line_no, line in enumerate(label_file.read_text().splitlines(), start=1):
            parts = line.split()
            if len(parts) != 5:
                problems.append(f'{label_file}:{line_no} expected 5 values')
                continue
            try:
                cls = int(parts[0])
                coords = [float(x) for x in parts[1:]]
            except ValueError:
                problems.append(f'{label_file}:{line_no} contains non-numeric values')
                continue
            if cls < 0 or cls >= len(CLASS_NAMES):
                problems.append(f'{label_file}:{line_no} invalid class id {cls}')
            if any(x < 0 or x > 1 for x in coords):
                problems.append(f'{label_file}:{line_no} coordinates outside [0,1]')
    return problems

for split in ['train', 'val', 'test']:
    problems = validate_yolo_labels(split)
    print(split, 'OK' if not problems else f'{len(problems)} problem(s)')
    for p in problems[:10]:
        print(' ', p)

## 3. Load a pretrained YOLO model

We use a pretrained YOLO checkpoint and fine-tune it on the poultry dataset. This is transfer learning: the model starts with useful visual features learned from a large general-purpose dataset rather than random initialization.


In [ ]:
MODEL_NAME = 'yolo11n.pt'
model = YOLO(MODEL_NAME)
print('Loaded:', MODEL_NAME)

## 4. Train the poultry detector

Start with a relatively small configuration. Once the pipeline works, increase epochs, image size, model size, or augmentation depending on dataset quality and GPU availability.


In [ ]:
TRAIN_EPOCHS = 50
IMAGE_SIZE = 640
BATCH_SIZE = 16

results = model.train(
    data=str(YAML_PATH),
    epochs=TRAIN_EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    patience=10,
    pretrained=True,
    project='runs/poultryvision',
    name='yolo_poultry_detector',
    exist_ok=True,
)

## 5. Validate the trained model

Important detection metrics include **precision, recall, mAP@50, and mAP@50-95**. For a farm-monitoring system, inspect class-level performance rather than looking only at one overall number.


In [ ]:
best_model_path = 'runs/poultryvision/yolo_poultry_detector/weights/best.pt'
trained_model = YOLO(best_model_path)

metrics = trained_model.val(
    data=str(YAML_PATH),
    imgsz=IMAGE_SIZE,
    split='val',
)

print(metrics)

## 6. Image inference

Set `IMAGE_PATH` to a new poultry image that was not used for training. The model will draw bounding boxes and confidence scores.


In [ ]:
IMAGE_PATH = '/content/test_poultry.jpg'

if os.path.exists(IMAGE_PATH):
    prediction = trained_model.predict(
        source=IMAGE_PATH,
        conf=0.35,
        imgsz=IMAGE_SIZE,
        save=True,
        project='runs/poultryvision',
        name='image_predictions',
        exist_ok=True,
    )
    annotated = prediction[0].plot()
    plt.figure(figsize=(12, 8))
    plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.show()
else:
    print('Upload an image and update IMAGE_PATH:', IMAGE_PATH)

## 7. Count detected objects in an image

This converts detection results into a simple farm metric: number of chickens, eggs, and chicks detected in one frame.


In [ ]:
def count_detections(result, class_names):
    counts = {name: 0 for name in class_names}
    if result.boxes is None:
        return counts

    class_ids = result.boxes.cls.cpu().numpy().astype(int)
    for class_id in class_ids:
        if 0 <= class_id < len(class_names):
            counts[class_names[class_id]] += 1
    return counts

if os.path.exists(IMAGE_PATH):
    result = trained_model.predict(source=IMAGE_PATH, conf=0.35, imgsz=IMAGE_SIZE, verbose=False)[0]
    counts = count_detections(result, CLASS_NAMES)
    print('Detected objects:', counts)

## 8. Video inference

For a real poultry farm, the next step is processing CCTV or webcam video. The example below reads a video file, detects objects frame-by-frame, and writes an annotated video.


In [ ]:
VIDEO_PATH = '/content/poultry_video.mp4'
OUTPUT_VIDEO = 'runs/poultryvision/poultry_annotated.mp4'

def process_video(model, video_path, output_path, conf=0.35):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise FileNotFoundError(f'Could not open video: {video_path}')

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)

    writer = cv2.VideoWriter(
        output_path,
        cv2.VideoWriter_fourcc(*'mp4v'),
        fps,
        (width, height),
    )

    frame_count = 0
    aggregate = {name: 0 for name in CLASS_NAMES}

    while True:
        ok, frame = cap.read()
        if not ok:
            break

        result = model.predict(source=frame, conf=conf, imgsz=IMAGE_SIZE, verbose=False)[0]
        annotated = result.plot()
        counts = count_detections(result, CLASS_NAMES)

        # Display current-frame counts on the video.
        y = 30
        for name, value in counts.items():
            cv2.putText(annotated, f'{name}: {value}', (15, y),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
            y += 30

        writer.write(annotated)
        frame_count += 1

        for name, value in counts.items():
            aggregate[name] += value

    cap.release()
    writer.release()
    return frame_count, aggregate

if os.path.exists(VIDEO_PATH):
    frames, aggregate = process_video(trained_model, VIDEO_PATH, OUTPUT_VIDEO)
    print('Processed frames:', frames)
    print('Aggregate frame detections:', aggregate)
    print('Saved:', OUTPUT_VIDEO)
else:
    print('Upload a video and update VIDEO_PATH:', VIDEO_PATH)

## 9. Real-time tracking — next upgrade

Detection alone can count the same chicken repeatedly across video frames. For true unique-object counting, use a tracker.

Recommended next step:

```text
YOLO Detection
      ↓
ByteTrack / BoT-SORT
      ↓
Persistent object IDs
      ↓
Line / zone crossing
      ↓
Unique chicken / egg count
```

Ultralytics supports tracking APIs that can be used as the next notebook stage. Do not interpret a simple sum of detections across frames as the number of unique animals or eggs.


## 10. Production project roadmap

```text
PoultryVision v1
YOLO detection
       ↓
v2
Detection + tracking
       ↓
v3
Chicken/egg counting
       ↓
v4
FastAPI inference API
       ↓
v5
React + TypeScript dashboard
       ↓
v6
WebSocket live camera stream
       ↓
v7
MongoDB analytics
       ↓
v8
Docker + CI/CD + cloud deployment
```

### Suggested API
```text
GET  /health
POST /predict
POST /detect-video
GET  /analytics/daily
GET  /analytics/production
```

### Suggested dashboard metrics
- Current chicken count
- Current chick count
- Current egg detections
- Daily egg-production count
- Hourly activity
- Camera status
- Detection confidence
- Alert history


## 11. Important modeling considerations

- Keep train/validation/test scenes separated where possible; frames from the same video should not be randomly split across all sets because that can cause leakage.
- Include different lighting, camera angles, animal sizes, occlusion, and backgrounds.
- Measure per-class precision/recall, not only overall mAP.
- Test on genuinely unseen farm/camera conditions.
- For unique counting, use tracking rather than summing frame detections.
- For health or welfare alerts, validate the model with appropriate domain experts before real-world use.

## Portfolio outcome

After completing this notebook, you will have demonstrated:

**Transfer Learning → YOLO → Object Detection → Evaluation → Video Inference → Counting → Tracking roadmap → Production AI architecture**
